# Adversarial Attacks on Image Classifiers

Neural networks are surprisingly fragile. A human can barely perceive the difference between a clean image and an adversarial one, yet the model's prediction can flip from correct to wildly wrong. This notebook builds up the theory from scratch, implements two classic attacks (FGSM and PGD), and demonstrates a localized patch attack. No prior adversarial ML knowledge assumed.

In [ ]:
# pip install torch torchvision matplotlib requests Pillow  # uncomment if needed

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
import io
import json
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Image Loading and Preprocessing

Before attacking any model we need a reliable preprocessing pipeline. The standard ImageNet pipeline resizes and center-crops to 224x224, converts to a float tensor in [0, 1], then normalizes each channel to zero mean and unit variance using the ImageNet statistics. We keep the raw [0, 1] tensor alongside the normalized version so we can display the image without reversing the normalization.

In [ ]:
# pip install torch torchvision matplotlib requests Pillow  # uncomment if needed

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
import io
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- ImageNet normalization constants ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Full preprocessing pipeline: resize -> crop -> tensor -> normalize
preprocess_pipeline = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),                                 # [0, 1], shape (C, H, W)
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Partial pipeline that stops before normalization (useful for perturbation in pixel space)
to_tensor_pipeline = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),   # [0, 1]
])

normalize_transform = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

def denormalize(tensor):
    """Reverse ImageNet normalization so the tensor can be displayed as an image."""
    mean = torch.tensor(IMAGENET_MEAN, device=tensor.device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=tensor.device).view(3, 1, 1)
    return (tensor * std + mean).clamp(0.0, 1.0)

def load_image_from_url(url):
    """Download a JPEG/PNG from a URL and return a (C, H, W) normalized tensor."""
    resp = requests.get(url, timeout=15)
    pil_img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    tensor = preprocess_pipeline(pil_img).unsqueeze(0)   # (1, C, H, W)
    return tensor.to(device)

# --- Load ImageNet labels ---
labels_url = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
try:
    imagenet_labels = json.loads(requests.get(labels_url, timeout=10).text)
except Exception:
    imagenet_labels = [str(i) for i in range(1000)]
    print('Could not download labels, using class indices.')

# --- Load ResNet-50 and test on one sample ---
resnet50_model = resnet50(weights=ResNet50_Weights.DEFAULT).to(device).eval()
print(f'ResNet-50 parameters: {sum(p.numel() for p in resnet50_model.parameters()):,}')

SAMPLE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg'
try:
    sample_input = load_image_from_url(SAMPLE_URL)
    with torch.no_grad():
        logits = resnet50_model(sample_input)
    probs     = torch.softmax(logits, dim=1)
    top_idx   = probs.argmax(dim=1).item()
    top_label = imagenet_labels[top_idx]
    top_conf  = probs[0, top_idx].item()
    print(f'ResNet-50 clean prediction: {top_label}  ({top_conf:.1%} confidence)')
    print('Preprocessing pipeline and ResNet-50 are ready.')
except Exception as e:
    print(f'Could not download sample image: {e}')
    print('Using a random tensor for subsequent cells.')
    sample_input = torch.randn(1, 3, 224, 224).to(device)
    top_idx   = 207   # golden retriever fallback
    top_label = imagenet_labels[top_idx]
    top_conf  = 0.0

## 1. The Threat Model

Before writing a single line of attack code, it helps to be precise about what the adversary controls and what they don't.

**What the adversary controls:**
- The input pixels at inference time. They can add a carefully crafted noise tensor to an image before feeding it to the model.

**What the adversary does NOT control:**
- The model weights. Those are fixed. The adversary can query the model (white-box: they know the architecture and weights; black-box: they can only see outputs).
- The training data, loss function, or any other training-time behavior.

This notebook focuses on **white-box attacks**: the adversary has full access to the model, including gradients. This is the strongest threat model and gives us the clearest math.

### Why do small perturbations fool models?

The short answer: a classifier's decision boundary is not smooth in high-dimensional pixel space.

Think about a 224x224 RGB image. That's 150,528 dimensions. The training data only covers a tiny fraction of this space. The model learns a decision boundary that works well on the data manifold (real images), but it has no reason to be well-behaved just off that manifold.

When you add a small, human-imperceptible perturbation, you step slightly off the data manifold. The model's learned features, especially texture-sensitive ones, can change drastically in response. The perturbation is designed to push the input across a decision boundary by moving in the direction that maximally increases the loss.

**The budget constraint:** We want the perturbation to be imperceptible. The standard way to enforce this is the L-infinity norm: every pixel can change by at most epsilon. Common values: `epsilon = 8/255` (nearly invisible) to `epsilon = 32/255` (slightly visible).

```
x_adv = x + delta   where  ||delta||_inf <= epsilon
```

In [ ]:
# Load a pretrained ResNet-18 and put it in eval mode
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model = model.to(device)
model.eval()

# Load the ImageNet class labels
labels_url = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
try:
    response = requests.get(labels_url, timeout=10)
    imagenet_labels = json.loads(response.text)
except Exception:
    # Fallback: just use class indices if download fails
    imagenet_labels = [str(i) for i in range(1000)]
    print('Could not download labels, using class indices instead')

print(f'Model loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Standard ImageNet preprocessing
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),  # converts to [0, 1]
])

normalize = T.Normalize(mean=imagenet_mean, std=imagenet_std)

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(imagenet_mean).view(3, 1, 1).to(tensor.device)
    std  = torch.tensor(imagenet_std).view(3, 1, 1).to(tensor.device)
    return (tensor * std + mean).clamp(0, 1)

def predict(x_normalized):
    """Run the model on a normalized batch and return (class_index, label, confidence)."""
    with torch.no_grad():
        logits = model(x_normalized)
    probs = torch.softmax(logits, dim=1)
    idx = probs.argmax(dim=1).item()
    return idx, imagenet_labels[idx], probs[0, idx].item()

def load_url_image(url):
    """Download an image from a URL and return a preprocessed tensor [1, 3, 224, 224] in [0,1]."""
    resp = requests.get(url, timeout=15)
    img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    return preprocess(img).unsqueeze(0)  # [1, 3, 224, 224]

# A few stable URLs of ImageNet-class images
SAMPLE_URLS = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/A_small_cup_of_coffee.JPG/320px-A_small_cup_of_coffee.JPG',
]

print('Preprocessing utilities defined.')

## 2. FGSM: Fast Gradient Sign Method

FGSM was introduced by Goodfellow et al. in 2014. The idea is beautifully simple.

**The objective:** Given an input `x` with true label `y`, find a perturbation `delta` such that:
1. The model predicts the wrong class on `x + delta`
2. `||delta||_inf <= epsilon`

**The derivation:** We want to *maximize* the loss `L(x + delta, y)` subject to `||delta||_inf <= epsilon`.

The gradient `grad_x L` tells us how the loss changes with respect to each pixel. To increase the loss as much as possible within the L-infinity ball, we take one step of size `epsilon` in the direction of the gradient's sign:

```
delta = epsilon * sign(grad_x L(x, y))
x_adv = clip(x + delta, 0, 1)
```

That's the entire algorithm. One forward pass to compute the loss, one backward pass to get the gradient, take the sign, multiply by epsilon.

The sign operation is key: it ensures every pixel gets exactly the same update magnitude (epsilon), which is the optimal single-step move under L-infinity constraints.

In [ ]:
def fgsm_attack(model, x, y, epsilon, loss_fn=nn.CrossEntropyLoss()):
    """
    Fast Gradient Sign Method.

    Args:
        model:   PyTorch model in eval mode
        x:       normalized input tensor [B, 3, H, W], requires no grad initially
        y:       true class labels [B], LongTensor
        epsilon: perturbation budget in normalized space
        loss_fn: classification loss (default: cross-entropy)

    Returns:
        x_adv:   adversarial example (same shape as x, normalized)
    """
    x_adv = x.clone().detach().requires_grad_(True).to(device)

    # Forward pass
    logits = model(x_adv)
    loss = loss_fn(logits, y)

    # Backward pass: compute gradient of loss w.r.t. input
    model.zero_grad()
    loss.backward()

    # Perturbation: step in the sign direction of the gradient
    grad_sign = x_adv.grad.sign()
    x_adv = x_adv.detach() + epsilon * grad_sign

    # No clipping to [0,1] here because we're in normalized space.
    # The denormalization handles display; validity is maintained by epsilon being small.
    return x_adv.detach()


def epsilon_to_normalized(epsilon_255, mean=imagenet_mean, std=imagenet_std):
    """
    Convert an epsilon specified in [0,255] pixel units to normalized space.
    Since normalization is per-channel, we return the average across channels
    as a conservative estimate.
    """
    eps_01 = epsilon_255 / 255.0
    # Scale by average std
    return eps_01 / np.mean(std)

print('FGSM defined.')

In [ ]:
# Download one test image and run FGSM on it
epsilon = 8 / 255.0  # budget in [0,1] pixel space
# Adjust for normalization: divide by std (approximate, using channel mean of stds)
epsilon_norm = epsilon / np.mean(imagenet_std)

try:
    x_raw = load_url_image(SAMPLE_URLS[0]).to(device)  # [1, 3, 224, 224], [0,1]
    x_norm = normalize(x_raw[0]).unsqueeze(0)          # normalized

    # Get the clean prediction
    clean_idx, clean_label, clean_conf = predict(x_norm)
    print(f'Clean prediction:       {clean_label} ({clean_conf:.1%})')

    # Run FGSM using the true label
    y_true = torch.tensor([clean_idx]).to(device)
    x_adv_norm = fgsm_attack(model, x_norm, y_true, epsilon_norm)

    # Evaluate adversarial example
    adv_idx, adv_label, adv_conf = predict(x_adv_norm)
    print(f'Adversarial prediction: {adv_label} ({adv_conf:.1%})')
    print(f'Attack succeeded: {adv_idx != clean_idx}')

    # Compute perturbation magnitude
    delta = (x_adv_norm - x_norm).abs()
    print(f'Max perturbation (normalized): {delta.max().item():.4f}')
    print(f'Mean perturbation (normalized): {delta.mean().item():.6f}')

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    img_clean = denormalize(x_norm[0]).permute(1, 2, 0).cpu().numpy()
    img_adv   = denormalize(x_adv_norm[0]).permute(1, 2, 0).cpu().numpy()
    img_delta = (delta[0] * 10).clamp(0, 1).permute(1, 2, 0).cpu().numpy()  # amplified for visibility

    axes[0].imshow(img_clean)
    axes[0].set_title(f'Clean\n{clean_label} ({clean_conf:.1%})')
    axes[0].axis('off')

    axes[1].imshow(img_delta)
    axes[1].set_title(f'Perturbation (10x amplified)\nmax={epsilon:.4f}')
    axes[1].axis('off')

    axes[2].imshow(img_adv)
    axes[2].set_title(f'Adversarial\n{adv_label} ({adv_conf:.1%})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig('fgsm_demo.png', dpi=100, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'Could not download image: {e}')
    print('Generating a random tensor for demonstration...')
    torch.manual_seed(42)
    x_norm = torch.randn(1, 3, 224, 224).to(device) * 0.5
    y_true = torch.tensor([207]).to(device)  # golden retriever
    x_adv_norm = fgsm_attack(model, x_norm, y_true, epsilon_norm)
    delta = (x_adv_norm - x_norm).abs()
    print(f'Max perturbation (normalized): {delta.max().item():.4f}')

### FGSM: Step-by-Step Code with Formula

The formula is:

```
x_adv = x + epsilon * sign( grad_x  L(f(x), y) )
```

Each line of the implementation below maps directly to one term in that formula.

In [ ]:
def fgsm_step_by_step(model, x_norm, y_true, epsilon):
    """
    FGSM implementation spelled out line by line.

    Formula:  x_adv = x + epsilon * sign( grad_x L(f(x), y) )

    Args:
        model:    PyTorch classifier in eval mode
        x_norm:   normalized input tensor, shape (1, 3, 224, 224)
        y_true:   ground-truth class index as a 1-element LongTensor
        epsilon:  perturbation magnitude (in normalized pixel units)

    Returns:
        x_adv:    adversarial example (same shape, normalized)
        delta:    the raw perturbation tensor (before adding to x)
        grad:     the input gradient (for inspection)
    """
    loss_fn = nn.CrossEntropyLoss()

    # Step 1 -- Enable gradient tracking on the input.
    #           We need d(loss)/d(x), not d(loss)/d(weights).
    x_adv = x_norm.clone().detach().requires_grad_(True)

    # Step 2 -- Forward pass: compute f(x), i.e. the model logits.
    logits = model(x_adv)           # shape: (1, 1000)

    # Step 3 -- Compute the classification loss L(f(x), y).
    loss = loss_fn(logits, y_true)  # scalar

    # Step 4 -- Backward pass: d(loss)/d(x).
    model.zero_grad()
    loss.backward()
    grad = x_adv.grad.clone()       # shape: (1, 3, 224, 224)

    # Step 5 -- Take the sign of the gradient.
    #           sign() maps each element to -1, 0, or +1.
    grad_sign = grad.sign()

    # Step 6 -- Scale by epsilon and ADD to the input.
    #           This is the single step of gradient ascent on the loss.
    delta = epsilon * grad_sign     # perturbation
    x_adv = (x_norm.detach() + delta).detach()

    return x_adv, delta, grad


# --- Run on the sample image loaded above ---
eps_pixel  = 8.0 / 255.0                          # budget in [0,1] pixel space
eps_norm   = eps_pixel / np.mean(IMAGENET_STD)    # scaled to normalized space

y_label = torch.tensor([top_idx], device=device)

x_adv_fgsm, delta_fgsm, grad_fgsm = fgsm_step_by_step(
    resnet50_model, sample_input, y_label, eps_norm
)

# --- Evaluate predictions ---
def predict_tensor(model, x_norm, labels):
    with torch.no_grad():
        probs = torch.softmax(model(x_norm), dim=1)
    idx  = probs.argmax(dim=1).item()
    return idx, labels[idx], probs[0, idx].item()

clean_idx,  clean_lbl,  clean_c  = predict_tensor(resnet50_model, sample_input, imagenet_labels)
adv_idx,    adv_lbl,    adv_c    = predict_tensor(resnet50_model, x_adv_fgsm,   imagenet_labels)

print(f'epsilon (pixel space):      {eps_pixel:.4f}  ({eps_pixel*255:.1f}/255)')
print(f'epsilon (normalized space): {eps_norm:.4f}')
print()
print(f'Clean prediction:       {clean_lbl} ({clean_c:.1%})')
print(f'Adversarial prediction: {adv_lbl}   ({adv_c:.1%})')
print(f'Attack succeeded:       {adv_idx != clean_idx}')
print()
print(f'Gradient stats:  min={grad_fgsm.min():.4f}  max={grad_fgsm.max():.4f}  '
      f'mean_abs={grad_fgsm.abs().mean():.6f}')
print(f'Delta stats:     min={delta_fgsm.min():.4f}  max={delta_fgsm.max():.4f}  '
      f'Linf={delta_fgsm.abs().max():.4f}')

### Visualization: Original | Perturbation | Adversarial

The perturbation is multiplied by 10 for display because at epsilon=8/255 the raw delta is invisible. The three-panel view makes it concrete: humans see identical images left and right, while the model's prediction can differ dramatically.

In [ ]:
def show_attack_triplet(x_clean, x_adv, delta, clean_title, adv_title,
                         amplify=10, save_path=None):
    """
    Display: original image | amplified perturbation | adversarial image.

    amplify: how much to scale the perturbation for visibility.
    """
    img_clean = denormalize(x_clean[0]).permute(1, 2, 0).cpu().numpy()
    img_adv   = denormalize(x_adv[0]).permute(1, 2, 0).cpu().numpy()

    # Amplify perturbation: shift to [0,1] center, then scale
    delta_vis = (delta[0].cpu().permute(1, 2, 0).numpy() * amplify + 0.5).clip(0, 1)

    linf_pixel = (delta[0].abs().max().item() * np.mean(IMAGENET_STD)) * 255

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(img_clean)
    axes[0].set_title(f'Original\n{clean_title}', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(delta_vis)
    axes[1].set_title(f'Perturbation ({amplify}x amplified)\nLinf = {linf_pixel:.1f}/255 pixels', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(img_adv)
    axes[2].set_title(f'Adversarial\n{adv_title}', fontsize=10)
    axes[2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


show_attack_triplet(
    sample_input, x_adv_fgsm, delta_fgsm,
    clean_title=f'{clean_lbl} ({clean_c:.1%})',
    adv_title=f'{adv_lbl} ({adv_c:.1%})',
    amplify=10,
    save_path='/tmp/fgsm_triplet.png',
)

## 3. PGD: Projected Gradient Descent

FGSM takes a single step. PGD (Madry et al., 2018) repeats the process: take a small step, then *project* back onto the epsilon ball. This is iterated projected gradient ascent on the loss.

**The algorithm:**

```
x_0 = x + uniform_noise(-epsilon, epsilon)   # random start inside the ball

for t in range(num_steps):
    grad = grad_x L(x_t, y)
    x_{t+1} = x_t + alpha * sign(grad)       # step size alpha << epsilon
    x_{t+1} = clip(x_{t+1}, x - epsilon, x + epsilon)  # project back
    x_{t+1} = clip(x_{t+1}, valid_range)     # keep in image bounds
```

**Why is PGD stronger than FGSM?**

FGSM is a single-step approximation. The loss landscape is not linear, so a single large step can overshoot the actual maximum inside the epsilon ball. PGD takes many smaller steps and projects after each one, so it finds a higher loss point within the budget.

The projection step is crucial: `clip(x_{t+1}, x - epsilon, x + epsilon)` ensures the adversarial example never exceeds the budget. Without this, iterating would just walk further and further away.

**Typical parameters:** `num_steps=40`, `alpha=2/255`. A common rule of thumb: `alpha = epsilon / (num_steps / 4)`.

In [ ]:
def pgd_attack(model, x, y, epsilon, alpha, num_steps, loss_fn=nn.CrossEntropyLoss(), random_start=True):
    """
    Projected Gradient Descent attack (Linf).

    Args:
        model:       PyTorch model in eval mode
        x:           normalized input [B, 3, H, W]
        y:           true labels [B]
        epsilon:     Linf budget (in normalized space)
        alpha:       step size per iteration
        num_steps:   number of gradient steps
        random_start: perturb with random noise before starting

    Returns:
        x_adv: adversarial example
    """
    x_adv = x.clone().detach().to(device)

    if random_start:
        # Start from a random point inside the epsilon ball
        noise = torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
        x_adv = x_adv + noise

    for step in range(num_steps):
        x_adv.requires_grad_(True)

        logits = model(x_adv)
        loss = loss_fn(logits, y)

        model.zero_grad()
        loss.backward()

        grad_sign = x_adv.grad.sign()

        # Gradient ascent step
        x_adv = x_adv.detach() + alpha * grad_sign

        # Project back onto the epsilon ball around the original x
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)

    return x_adv.detach()


print('PGD defined.')

In [ ]:
# Compare FGSM vs PGD on the same image
epsilon_px   = 8.0 / 255.0          # budget in pixel space [0,1]
epsilon_n    = epsilon_px / np.mean(imagenet_std)  # normalized
alpha_n      = (2.0 / 255.0) / np.mean(imagenet_std)
num_steps    = 40

try:
    # Use the image loaded above; fall back to a dummy tensor
    if 'x_norm' not in dir():
        raise NameError

    # FGSM
    x_fgsm = fgsm_attack(model, x_norm, y_true, epsilon_n)
    fgsm_idx, fgsm_label, fgsm_conf = predict(x_fgsm)

    # PGD
    x_pgd = pgd_attack(model, x_norm, y_true, epsilon_n, alpha_n, num_steps)
    pgd_idx, pgd_label, pgd_conf = predict(x_pgd)

    # Check budget compliance
    fgsm_budget = (x_fgsm - x_norm).abs().max().item()
    pgd_budget  = (x_pgd  - x_norm).abs().max().item()

    print(f'Epsilon budget (normalized): {epsilon_n:.4f}')
    print()
    print(f'Clean:  {clean_label} ({clean_conf:.1%})')
    print(f'FGSM:   {fgsm_label} ({fgsm_conf:.1%})  | budget used: {fgsm_budget:.4f} | fooled: {fgsm_idx != clean_idx}')
    print(f'PGD:    {pgd_label} ({pgd_conf:.1%})   | budget used: {pgd_budget:.4f} | fooled: {pgd_idx != clean_idx}')

    # Visual comparison
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    imgs = [
        (x_norm,  f'Clean\n{clean_label}\n({clean_conf:.1%})'),
        (x_fgsm,  f'FGSM (eps=8/255)\n{fgsm_label}\n({fgsm_conf:.1%})'),
        (x_pgd,   f'PGD-40 (eps=8/255)\n{pgd_label}\n({pgd_conf:.1%})'),
    ]
    for ax, (t, title) in zip(axes, imgs):
        img = denormalize(t[0]).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('fgsm_vs_pgd.png', dpi=100, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'Skipping visual comparison: {e}')

## Attack Strength Sweep: Accuracy vs. Epsilon

How does the model's accuracy degrade as we increase the perturbation budget? We run FGSM at six epsilon values on the sample image and record whether the attack succeeds, then plot the result. For a single image the outcome is binary (0 or 1), so we also track the model's confidence in its original class to see the degradation more smoothly.

In [ ]:
import torch.nn.functional as F

# Epsilon values in pixel space [0, 1]
epsilon_values = [0.0, 0.01, 0.02, 0.05, 0.1, 0.3]

sweep_results = []

y_label = torch.tensor([top_idx], device=device)
loss_fn  = nn.CrossEntropyLoss()

for eps_px in epsilon_values:
    eps_n = eps_px / np.mean(IMAGENET_STD)   # convert to normalized space

    if eps_n == 0.0:
        x_test = sample_input.clone()
    else:
        # FGSM
        x_tmp = sample_input.clone().detach().requires_grad_(True)
        loss  = loss_fn(resnet50_model(x_tmp), y_label)
        resnet50_model.zero_grad()
        loss.backward()
        x_test = (x_tmp.detach() + eps_n * x_tmp.grad.sign()).detach()

    with torch.no_grad():
        probs = torch.softmax(resnet50_model(x_test), dim=1)

    pred_idx  = probs.argmax(dim=1).item()
    orig_conf = probs[0, top_idx].item()     # confidence in the original class
    pred_lbl  = imagenet_labels[pred_idx]
    correct   = int(pred_idx == top_idx)

    sweep_results.append({
        'eps_px':    eps_px,
        'eps_255':   int(round(eps_px * 255)),
        'correct':   correct,
        'orig_conf': orig_conf,
        'pred':      pred_lbl,
    })
    print(f'eps={eps_px:.3f} ({int(eps_px*255):3d}/255)  '
          f'pred={pred_lbl:<20}  orig_class_conf={orig_conf:.3f}  correct={bool(correct)}')

# --- Plot: confidence in original class vs. epsilon ---
epsilons_255 = [r['eps_255'] for r in sweep_results]
orig_confs   = [r['orig_conf'] for r in sweep_results]
correct_vals = [r['correct']   for r in sweep_results]

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(epsilons_255, orig_confs, 'o-', color='steelblue', label='Confidence in original class')
ax1.set_xlabel('Epsilon (pixel units, /255)')
ax1.set_ylabel('Model confidence in original class', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(0, 1.05)

ax2 = ax1.twinx()
ax2.step(epsilons_255, correct_vals, where='post', color='firebrick',
         linestyle='--', alpha=0.7, label='Correct (1) / Fooled (0)')
ax2.set_ylabel('Correct prediction (1=yes, 0=no)', color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')
ax2.set_ylim(-0.1, 1.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

ax1.set_title('FGSM attack: confidence and correctness vs. epsilon\n(ResNet-50, single image)')
plt.tight_layout()
plt.savefig('/tmp/epsilon_sweep.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Patch Attacks

Both FGSM and PGD distribute the perturbation across the entire image. Patch attacks are different: they restrict the perturbation to a *localized rectangular region*, but within that region the perturbation is unconstrained (no L-infinity budget).

This matters in practice because a patch attack can be printed out and placed in the physical world. A sticker on a stop sign, a pattern on a T-shirt, a printed rectangle held in front of a camera.

**How it works:**
1. Define a fixed patch location and size (e.g., a 50x50 square in the top-left corner).
2. Optimize the patch contents to maximize misclassification. The rest of the image stays clean.
3. The patch is unconstrained in pixel value (clamped to [0,1]), so it can look like anything.

The optimization is the same as PGD, except the gradient update only applies to the patch region.

In [ ]:
def patch_attack(model, x, y, patch_size=50, num_steps=100, alpha=0.01,
                 patch_row=10, patch_col=10, loss_fn=nn.CrossEntropyLoss()):
    """
    Localized patch attack.

    Optimizes a patch of shape (3, patch_size, patch_size) placed at
    (patch_row, patch_col) in the normalized image. The rest of the image
    is unchanged.

    Returns:
        x_patched: the image with the adversarial patch applied
        patch:     the optimized patch tensor
    """
    x = x.clone().detach().to(device)
    r, c = patch_row, patch_col
    pr, pc = patch_size, patch_size

    # Initialize the patch from the original image region (warm start)
    patch = x[:, :, r:r+pr, c:c+pc].clone().detach().requires_grad_(True)

    optimizer = torch.optim.Adam([patch], lr=alpha)

    for step in range(num_steps):
        # Apply patch to a fresh copy of x
        x_patched = x.clone()
        x_patched[:, :, r:r+pr, c:c+pc] = patch

        logits = model(x_patched)
        # Maximize loss = minimize negative loss
        loss = -loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Clamp patch to valid normalized range (approximate, based on typical norm range)
        with torch.no_grad():
            patch.clamp_(-3.0, 3.0)

        if (step + 1) % 25 == 0:
            probs = torch.softmax(logits.detach(), dim=1)
            pred = probs.argmax(dim=1).item()
            print(f'  Step {step+1:3d}: loss={-loss.item():.4f}, '
                  f'pred={imagenet_labels[pred]} ({probs[0,pred]:.1%})')

    # Build final patched image
    x_patched = x.clone()
    with torch.no_grad():
        x_patched[:, :, r:r+pr, c:c+pc] = patch

    return x_patched.detach(), patch.detach()


print('Patch attack defined.')

In [ ]:
# Run the patch attack
try:
    print('Running patch attack (100 steps)...')
    x_patched, opt_patch = patch_attack(
        model, x_norm, y_true,
        patch_size=50, num_steps=100, alpha=0.02,
        patch_row=10, patch_col=10
    )

    patch_idx, patch_label, patch_conf = predict(x_patched)
    print(f'\nClean:         {clean_label} ({clean_conf:.1%})')
    print(f'After patch:   {patch_label} ({patch_conf:.1%})')
    print(f'Attack succeeded: {patch_idx != clean_idx}')

    # What fraction of the image does the patch cover?
    patch_frac = (50 * 50) / (224 * 224)
    print(f'Patch covers {patch_frac:.1%} of the image')

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    img_clean   = denormalize(x_norm[0]).permute(1, 2, 0).cpu().numpy()
    img_patched = denormalize(x_patched[0]).permute(1, 2, 0).cpu().numpy()
    img_patch   = denormalize(opt_patch[0]).permute(1, 2, 0).cpu().numpy()

    axes[0].imshow(img_clean)
    axes[0].set_title(f'Clean\n{clean_label} ({clean_conf:.1%})')
    axes[0].axis('off')

    axes[1].imshow(img_patch)
    axes[1].set_title('Optimized patch\n(50x50 pixels)')
    axes[1].axis('off')

    axes[2].imshow(img_patched)
    # Draw a rectangle to show the patch location
    rect = patches.Rectangle((10, 10), 50, 50, linewidth=2, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].set_title(f'Patched image\n{patch_label} ({patch_conf:.1%})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig('patch_attack.png', dpi=100, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'Patch attack demo skipped: {e}')

## 5. Hands-on: Comparing Attack Success Rates

Now we run a systematic comparison. We take 5 sample images, run both FGSM and PGD at `epsilon=8/255`, and measure:
- Clean accuracy (what the model predicts without any attack)
- FGSM adversarial accuracy (how often the attack fools the model)
- PGD adversarial accuracy (same)

We also try a second epsilon (`epsilon=16/255`) to see how the attack budget affects success rate.

In [ ]:
# We'll generate synthetic test images using torchvision's built-in examples
# and also download a few from Wikipedia if available.

# More stable image URLs (Wikipedia CC images)
test_urls = [
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg', 'dog'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg', 'cat'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/f/f3/Flamingos_Laguna_Colorada.jpg/320px-Flamingos_Laguna_Colorada.jpg', 'flamingo'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/320px-Dog_Breeds.jpg', 'dog2'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg', 'ant'),
]

def eval_attacks(urls, epsilons=[8/255, 16/255]):
    results = {eps: {'clean': 0, 'fgsm_fooled': 0, 'pgd_fooled': 0, 'total': 0} for eps in epsilons}

    loaded_images = []
    for url, name in urls:
        try:
            x_raw = load_url_image(url).to(device)
            x_n = normalize(x_raw[0]).unsqueeze(0)
            idx, label, conf = predict(x_n)
            loaded_images.append((x_n, idx, label, conf, name))
            print(f'  Loaded {name}: predicted {label} ({conf:.1%})')
        except Exception as e:
            print(f'  Skipped {name}: {e}')

    if not loaded_images:
        # Fall back to random synthetic images
        print('No images loaded, using random tensors...')
        torch.manual_seed(0)
        for i in range(5):
            x_n = torch.randn(1, 3, 224, 224).to(device) * 0.3
            idx, label, conf = predict(x_n)
            loaded_images.append((x_n, idx, label, conf, f'synthetic_{i}'))

    print(f'\nEvaluating attacks on {len(loaded_images)} images...')
    all_rows = []

    for x_n, clean_idx, clean_label, clean_conf, name in loaded_images:
        y_t = torch.tensor([clean_idx]).to(device)
        row = {'name': name, 'clean': clean_label, 'clean_conf': clean_conf}

        for eps in epsilons:
            eps_n = eps / np.mean(imagenet_std)
            alpha_n = (2/255) / np.mean(imagenet_std)

            x_fgsm = fgsm_attack(model, x_n, y_t, eps_n)
            x_pgd  = pgd_attack(model, x_n, y_t, eps_n, alpha_n, num_steps=40)

            fgsm_idx, fgsm_lbl, fgsm_c = predict(x_fgsm)
            pgd_idx,  pgd_lbl,  pgd_c  = predict(x_pgd)

            row[f'fgsm_eps{int(eps*255)}'] = fgsm_lbl
            row[f'fgsm_fooled_{int(eps*255)}'] = (fgsm_idx != clean_idx)
            row[f'pgd_eps{int(eps*255)}'] = pgd_lbl
            row[f'pgd_fooled_{int(eps*255)}'] = (pgd_idx != clean_idx)

        all_rows.append(row)

    return all_rows, loaded_images

print('Loading and evaluating...')
rows, images = eval_attacks(test_urls)

In [ ]:
# Print a summary table
import pandas as pd

if rows:
    df = pd.DataFrame(rows)
    print('Per-image results:')
    print(df.to_string(index=False))

    print('\n--- Attack Success Rates ---')
    n = len(rows)
    for eps in [8, 16]:
        fgsm_rate = sum(r.get(f'fgsm_fooled_{eps}', False) for r in rows) / n
        pgd_rate  = sum(r.get(f'pgd_fooled_{eps}',  False) for r in rows) / n
        print(f'epsilon={eps}/255:  FGSM fool rate={fgsm_rate:.0%}   PGD fool rate={pgd_rate:.0%}')

    # Bar chart
    fig, ax = plt.subplots(figsize=(7, 4))
    epsilons = [8, 16]
    fgsm_rates = [sum(r.get(f'fgsm_fooled_{e}', False) for r in rows)/n for e in epsilons]
    pgd_rates  = [sum(r.get(f'pgd_fooled_{e}',  False) for r in rows)/n for e in epsilons]

    x = np.arange(len(epsilons))
    width = 0.3
    ax.bar(x - width/2, fgsm_rates, width, label='FGSM', color='steelblue')
    ax.bar(x + width/2, pgd_rates,  width, label='PGD-40', color='firebrick')
    ax.set_xticks(x)
    ax.set_xticklabels([f'eps={e}/255' for e in epsilons])
    ax.set_ylabel('Attack success rate')
    ax.set_title('FGSM vs PGD attack success rate on ResNet-18')
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig('attack_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()

## Key Takeaways

**FGSM** is a one-shot attack. Fast to compute, but it takes a single large step and can overshoot the optimal perturbation. At small epsilon it may fail where PGD succeeds.

**PGD** iterates, projecting back to the budget after each step. It finds a higher-loss point within the same budget. At the same epsilon, PGD almost always achieves higher attack success rates than FGSM.

**Patch attacks** are qualitatively different: they use unlimited perturbation in a small area. This makes them physically realizable but also more visible.

**Epsilon matters a lot.** Going from 8/255 to 16/255 often doubles or triples the attack success rate. The adversarial perturbations at 8/255 are genuinely imperceptible to humans, yet they reliably fool state-of-the-art models.

**White-box assumption.** Everything here assumed the attacker knows the model weights and can compute gradients. In practice, attacks often *transfer*: an adversarial example crafted for one model also fools a different model trained on the same data. This is called transferability, and it is why defenses need to be evaluated carefully.

## AutoAttack: A Reliable Benchmark Attack

FGSM and PGD are easy to implement but can be beaten by defenses that exploit their specific failure modes (e.g., gradient masking). AutoAttack (Croce and Hein, 2020) combines four diverse attacks into one reliable ensemble and has become the standard benchmark for evaluating adversarial defenses.

Install with: `pip install autoattack`

The API is straightforward: wrap your model, call `run_standard_evaluation`, get adversarial examples back.

In [ ]:
# pip install autoattack   # uncomment to install

try:
    from autoattack import AutoAttack

    # AutoAttack expects inputs in [0, 1] (not normalized).
    # We create a wrapper that normalizes inside the forward pass.
    class NormalizedModel(torch.nn.Module):
        def __init__(self, model, mean, std):
            super().__init__()
            self.model = model
            self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
            self.register_buffer('std',  torch.tensor(std).view(1, 3, 1, 1))

        def forward(self, x):
            return self.model((x - self.mean) / self.std)

    norm_model = NormalizedModel(resnet50_model, IMAGENET_MEAN, IMAGENET_STD).to(device).eval()

    # Build a tiny batch: one clean image in [0,1] + its label
    # load_image_from_url returns a normalized tensor; we reverse normalization here.
    mean_t = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std_t  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1)
    x_01   = (sample_input * std_t + mean_t).clamp(0, 1)   # [0,1] tensor (1, 3, 224, 224)
    labels_aa = torch.tensor([top_idx], device=device)

    # AutoAttack with Linf norm, eps = 8/255
    adversary = AutoAttack(
        norm_model,
        norm='Linf',
        eps=8.0 / 255.0,
        version='standard',    # uses APGD-CE, APGD-T, FAB-T, Square
        device=device,
        verbose=True,
    )

    x_adv_aa = adversary.run_standard_evaluation(x_01, labels_aa, bs=1)

    # Evaluate
    with torch.no_grad():
        pred_aa = norm_model(x_adv_aa).argmax(dim=1).item()
    print(f'\nAutoAttack result: {imagenet_labels[pred_aa]}  (original: {imagenet_labels[top_idx]})')
    print(f'Attack succeeded: {pred_aa != top_idx}')

except ImportError:
    print('autoattack not installed. Run:  pip install autoattack')
    print()
    print('Example code that would run:')
    print("""
    from autoattack import AutoAttack

    adversary = AutoAttack(
        model,          # model that takes [0,1] inputs
        norm='Linf',
        eps=8/255,
        version='standard',
    )
    x_adv = adversary.run_standard_evaluation(x_clean_01, y_true, bs=32)
    """)
except Exception as e:
    print(f'AutoAttack demo failed: {e}')

## Patch Attack: 32x32 Adversarial Patch

A patch attack concentrates all the perturbation budget in a small region. Within that region the pixel values are completely unconstrained (clamped to valid range, not limited by an Linf budget). The patch is physically realizable: you can print it and hold it in front of a camera.

Below we optimize a 32x32 patch at a fixed location using gradient ascent with Adam, then show the image before and after applying the patch.

In [ ]:
import matplotlib.patches as mpatches

def patch_attack_32(model, x_norm, y_true,
                    patch_size=32,
                    patch_row=20, patch_col=20,
                    num_steps=150, lr=0.05,
                    loss_fn=nn.CrossEntropyLoss()):
    """
    Optimize a 32x32 adversarial patch placed at (patch_row, patch_col).

    The patch is unconstrained within normalized image bounds.
    The rest of the image is unchanged.

    Returns:
        x_patched:  full image with patch applied (normalized)
        patch:      the optimized patch tensor (3, patch_size, patch_size)
    """
    pr, pc = patch_row, patch_col
    ps     = patch_size
    x      = x_norm.clone().detach().to(device)

    # Initialize patch from the image region (warm start is faster to converge)
    patch = x[:, :, pr:pr+ps, pc:pc+ps].clone().requires_grad_(True)
    optimizer = torch.optim.Adam([patch], lr=lr)

    for step in range(num_steps):
        x_p = x.clone()
        x_p[:, :, pr:pr+ps, pc:pc+ps] = patch

        logits = model(x_p)
        loss   = -loss_fn(logits, y_true)   # negate: we want to maximize the loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Clamp to the typical normalized range (approximately -2.5 to 2.5)
        with torch.no_grad():
            patch.clamp_(-2.5, 2.5)

        if (step + 1) % 50 == 0:
            with torch.no_grad():
                pred = model(x_p).argmax(dim=1).item()
            print(f'  step {step+1:3d}: loss={-loss.item():.4f}  pred={imagenet_labels[pred]}')

    # Build final patched image
    x_final = x.clone()
    with torch.no_grad():
        x_final[:, :, pr:pr+ps, pc:pc+ps] = patch.detach()

    return x_final.detach(), patch.detach()


print('Running 32x32 patch attack (150 steps)...')
x_patched32, opt_patch32 = patch_attack_32(
    resnet50_model, sample_input, y_label,
    patch_size=32, patch_row=20, patch_col=20,
    num_steps=150, lr=0.05,
)

patch_pred_idx, patch_pred_lbl, patch_pred_c = predict_tensor(
    resnet50_model, x_patched32, imagenet_labels
)
print(f'\nBefore patch:  {clean_lbl} ({clean_c:.1%})')
print(f'After patch:   {patch_pred_lbl} ({patch_pred_c:.1%})')
print(f'Attack succeeded: {patch_pred_idx != top_idx}')
print(f'Patch covers {32*32/(224*224):.1%} of the image')

# --- Visualization ---
import matplotlib.patches as mpl_patches

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

img_orig    = denormalize(sample_input[0]).permute(1, 2, 0).cpu().numpy()
img_patched = denormalize(x_patched32[0]).permute(1, 2, 0).cpu().numpy()
img_patch   = denormalize(opt_patch32[0]).permute(1, 2, 0).cpu().numpy()

axes[0].imshow(img_orig)
axes[0].set_title(f'Original\n{clean_lbl} ({clean_c:.1%})')
axes[0].axis('off')

axes[1].imshow(img_patch)
axes[1].set_title('Optimized 32x32 patch\n(unconstrained in content)')
axes[1].axis('off')

axes[2].imshow(img_patched)
rect = mpl_patches.Rectangle((20, 20), 32, 32,
                               linewidth=2, edgecolor='red', facecolor='none')
axes[2].add_patch(rect)
axes[2].set_title(f'Image with patch applied\n{patch_pred_lbl} ({patch_pred_c:.1%})')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/tmp/patch_attack_32.png', dpi=100, bbox_inches='tight')
plt.show()

## Transferability: Does the Attack Cross Models?

Adversarial examples crafted for ResNet-50 often fool other models even though those models never saw the perturbation during generation. This is called *transferability* and is a key reason adversarial attacks are threatening in black-box settings.

We generate FGSM and PGD adversarial examples using ResNet-50 as the source model, then evaluate those same examples on MobileNetV3-Large, which has a completely different architecture.

In [ ]:
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# Load MobileNetV3-Large as the target (transfer) model
mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT).to(device).eval()
print(f'MobileNetV3-Large parameters: {sum(p.numel() for p in mobilenet.parameters()):,}')

# --- Generate adversarial examples on ResNet-50 (source model) ---
eps_transfer = 8.0 / 255.0 / np.mean(IMAGENET_STD)   # normalized epsilon

# FGSM on ResNet-50
x_adv_transfer_fgsm, _, _ = fgsm_step_by_step(
    resnet50_model, sample_input, y_label, eps_transfer
)

# PGD-40 on ResNet-50 (reuse the pgd_attack function from the earlier cell)
def pgd_linf(model, x, y, epsilon, alpha, num_steps, loss_fn=nn.CrossEntropyLoss()):
    x_adv = x.clone().detach()
    noise = torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
    x_adv = x_adv + noise
    for _ in range(num_steps):
        x_adv.requires_grad_(True)
        loss = loss_fn(model(x_adv), y)
        model.zero_grad()
        loss.backward()
        x_adv = (x_adv.detach() + alpha * x_adv.grad.sign()).detach()
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
    return x_adv.detach()

alpha_transfer = (2.0 / 255.0) / np.mean(IMAGENET_STD)
x_adv_transfer_pgd = pgd_linf(
    resnet50_model, sample_input, y_label,
    eps_transfer, alpha_transfer, num_steps=40
)

# --- Evaluate on both models ---
print('\n--- Predictions on clean image ---')
for name, mdl in [('ResNet-50', resnet50_model), ('MobileNetV3-L', mobilenet)]:
    idx, lbl, c = predict_tensor(mdl, sample_input, imagenet_labels)
    print(f'  {name:<15}: {lbl} ({c:.1%})')

print('\n--- Predictions on FGSM adversarial (crafted on ResNet-50) ---')
for name, mdl in [('ResNet-50', resnet50_model), ('MobileNetV3-L', mobilenet)]:
    idx, lbl, c = predict_tensor(mdl, x_adv_transfer_fgsm, imagenet_labels)
    fooled = idx != top_idx
    print(f'  {name:<15}: {lbl} ({c:.1%})  fooled={fooled}')

print('\n--- Predictions on PGD-40 adversarial (crafted on ResNet-50) ---')
for name, mdl in [('ResNet-50', resnet50_model), ('MobileNetV3-L', mobilenet)]:
    idx, lbl, c = predict_tensor(mdl, x_adv_transfer_pgd, imagenet_labels)
    fooled = idx != top_idx
    print(f'  {name:<15}: {lbl} ({c:.1%})  fooled={fooled}')

print()
print('If MobileNetV3-L is fooled even though it was never the source model,')
print("that's transferability at work.")

## Exercise

Work through these tasks:

**Exercise 1: Epsilon sweep**
Run FGSM at epsilon values of 2/255, 4/255, 8/255, 16/255, and 32/255 on a single image. Plot the attack success rate (0 or 1 per image isn't that interesting, so print the confidence the model assigns to the original class). At what epsilon does the image start to look noticeably different to you?

**Exercise 2: PGD step count**
Fix epsilon=8/255 and run PGD with `num_steps` in [1, 5, 10, 20, 40, 100]. Plot the loss value after the final step as a function of `num_steps`. Does the loss keep increasing, or does it plateau?

**Exercise 3: Targeted attack**
The attacks above are *untargeted*: they just push the model away from the correct class. A *targeted attack* pushes toward a specific wrong class. Modify `fgsm_attack` to accept a `target_label` argument. Instead of maximizing `L(x, y_true)`, minimize `L(x, y_target)`. Try to make the model classify a dog as a cat.

Hint: for a targeted FGSM, the sign of the perturbation flips:
```python
delta = -epsilon * sign(grad_x L(x, y_target))
```

**Exercise 4: Transferability**
Generate adversarial examples using ResNet-18. Then load a different architecture (e.g., `torchvision.models.mobilenet_v2(weights='DEFAULT')`) and evaluate those same adversarial examples on it. What is the fool rate? Does the attack transfer?

## Exercise: Carlini-Wagner L2 Attack

The Carlini-Wagner (CW) attack finds the smallest L2 perturbation that causes misclassification, by solving an optimization problem directly rather than following the gradient sign. It is more expensive than FGSM or PGD but produces smaller, higher-quality perturbations.

You can implement it from scratch (the optimization approach below) or use the `foolbox` library, which provides a clean API for many attack algorithms.

In [ ]:
# pip install foolbox  # uncomment to use the foolbox option

# --- Option A: Implement the CW L2 attack from scratch ---
#
# The CW L2 formulation minimizes:
#   ||delta||_2^2  +  c * f(x + delta)
#
# where f is an "adversarial loss" that is <= 0 when the example is adversarial.
# The canonical f for untargeted attacks is:
#   f(x') = max(Z[y_true] - max_{j != y_true} Z[j], -kappa)
# Z are the model logits, kappa is a confidence margin (0 = just fool, >0 = fool confidently).
#
# We optimize delta using the tanh parameterization to keep x+delta in a valid range.

def cw_l2_attack(model, x_norm, y_true, c=1.0, kappa=0.0, num_steps=500, lr=0.01):
    """
    Simplified CW L2 attack (untargeted).

    Args:
        model:      PyTorch model in eval mode
        x_norm:     normalized input (1, 3, 224, 224)
        y_true:     ground-truth label as a 1-element LongTensor
        c:          trade-off constant (higher -> more adversarial, larger perturbation)
        kappa:      confidence margin
        num_steps:  optimization steps
        lr:         Adam learning rate

    Returns:
        x_adv:  adversarial example (normalized)
        l2_dist: L2 distance of the perturbation
    """
    # YOUR CODE HERE
    #
    # Hints:
    # 1. Use the tanh parameterization: x = 0.5 * (tanh(w) + 1), optimize w.
    #    This keeps x in [0,1] automatically (before normalization).
    # 2. The adversarial loss for untargeted attack:
    #       logits = model(x_adv)
    #       real   = logits[0, y_true]
    #       other  = max of logits over all j != y_true
    #       f_val  = max(real - other, -kappa)
    # 3. Total loss = ||delta||_2^2 + c * f_val
    # 4. Check every step if f_val <= 0 (attack succeeded); track the best x_adv
    #    that succeeded (the one with smallest L2 distance).
    raise NotImplementedError


# --- Option B: Use foolbox ---
#
# pip install foolbox
#
# import foolbox as fb
#
# fmodel = fb.PyTorchModel(resnet50_model, bounds=(-3, 3))  # normalized space
# attack = fb.attacks.L2CarliniWagnerAttack(steps=500)
#
# y_fb = y_label  # LongTensor
# _, x_adv_cw, success = attack(fmodel, sample_input, y_fb, epsilons=None)
# print(f'CW L2 succeeded: {success.item()}')
# l2 = (x_adv_cw - sample_input).norm(p=2).item()
# print(f'L2 perturbation: {l2:.4f}')


print('Exercise: implement cw_l2_attack above, or use the foolbox snippet.')
print('Reference: Carlini & Wagner, "Evaluating the Robustness of Neural Networks", 2017.')